In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. Загрузка датасета diamonds
# ============================================
print("📥 Загрузка датасета diamonds...")
df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv')
print(f"Размер датасета: {df.shape}")

# ============================================
# 2. Разделение на признаки и целевую переменную
# ============================================
target = 'price'
X = df.drop(target, axis=1)
y = df[target]

# ============================================
# 3. Определение категориальных и числовых признаков
# ============================================
cat_features = ['cut', 'color', 'clarity']
num_features = ['carat', 'depth', 'table', 'x', 'y', 'z']

# ============================================
# 4. Создание новых признаков (feature engineering)
# ============================================
print("➕ Создание новых признаков...")

# Соотношение сторон (защита от деления на 0)
X['aspect_ratio'] = X['x'] / (X['y'] + 0.001)

# Объём
X['volume'] = X['x'] * X['y'] * X['z']

# Площадь стола
X['table_area'] = (X['table'] / 100) * (X['x'] * X['y'])

# Соотношение глубины
X['depth_ratio'] = X['depth'] / (X['z'] + 0.001)

# Обновляем список числовых признаков
new_features = ['aspect_ratio', 'volume', 'table_area', 'depth_ratio']
num_features = num_features + new_features

print(f"Числовые признаки: {num_features}")
print(f"Категориальные признаки: {cat_features}")

# ============================================
# 5. Предобработка и масштабирование
# ============================================
print("⚙️ Предобработка данных...")

# Создаём пайплайн предобработки
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_features)
])

# Применяем предобработку
X_processed = preprocessor.fit_transform(X)

# Сохраняем препроцессор
joblib.dump(preprocessor, 'preprocessor.joblib')
print("✅ Препроцессор сохранён в preprocessor.joblib")

# ============================================
# 6. Преобразование целевой переменной (PowerTransformer)
# ============================================
print("⚙️ Преобразование целевой переменной...")
power_trans = PowerTransformer()
y_transformed = power_trans.fit_transform(y.values.reshape(-1, 1)).ravel()

# Сохраняем PowerTransformer
joblib.dump(power_trans, 'power_transformer.joblib')
print("✅ PowerTransformer сохранён в power_transformer.joblib")

# ============================================
# 7. Разделение на train/test
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_transformed, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# ============================================
# 8. Обучение модели
# ============================================
print("🧠 Обучение модели RandomForestRegressor...")
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ============================================
# 9. Оценка модели
# ============================================
y_pred_transformed = model.predict(X_test)
y_pred = power_trans.inverse_transform(y_pred_transformed.reshape(-1, 1)).ravel()
y_true = power_trans.inverse_transform(y_test.reshape(-1, 1)).ravel()

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"\n📊 Результаты модели:")
print(f"   RMSE: {rmse:.2f}")
print(f"   MAE: {mae:.2f}")
print(f"   R²: {r2:.4f}")

# ============================================
# 10. Сохранение модели
# ============================================
joblib.dump(model, 'diamonds_model.joblib')
print("✅ Модель сохранена в diamonds_model.joblib")

# ============================================
# 11. Сохранение списка колонок (важно для API)
# ============================================
# Получаем имена колонок после One-Hot Encoding
ohe = preprocessor.named_transformers_['cat']
feature_names = num_features.copy()

for i, cat in enumerate(cat_features):
    categories = ohe.categories_[i]
    # Пропускаем первую категорию (drop='first')
    for category in categories[1:]:
        feature_names.append(f"{cat}_{category}")

# Сохраняем список колонок
import json
with open('feature_columns.json', 'w') as f:
    json.dump(feature_names, f, indent=2)

print(f"✅ Список колонок сохранён в feature_columns.json")
print(f"   Всего колонок: {len(feature_names)}")

print("\n🎉 Готово! Файлы созданы:")
print("   - diamonds_model.joblib")
print("   - power_transformer.joblib")
print("   - preprocessor.joblib")
print("   - feature_columns.json")

📥 Загрузка датасета diamonds...
Размер датасета: (53940, 10)
➕ Создание новых признаков...
Числовые признаки: ['carat', 'depth', 'table', 'x', 'y', 'z', 'aspect_ratio', 'volume', 'table_area', 'depth_ratio']
Категориальные признаки: ['cut', 'color', 'clarity']
⚙️ Предобработка данных...
✅ Препроцессор сохранён в preprocessor.joblib
⚙️ Преобразование целевой переменной...
✅ PowerTransformer сохранён в power_transformer.joblib
Train size: (43152, 27), Test size: (10788, 27)
🧠 Обучение модели RandomForestRegressor...

📊 Результаты модели:
   RMSE: 757.62
   MAE: 361.58
   R²: 0.9639
✅ Модель сохранена в diamonds_model.joblib
✅ Список колонок сохранён в feature_columns.json
   Всего колонок: 27

🎉 Готово! Файлы созданы:
   - diamonds_model.joblib
   - power_transformer.joblib
   - preprocessor.joblib
   - feature_columns.json
